## Importing libraries and observe input folders in Kaggle notebook

In [ ]:
# Linear algebra
import numpy as np

# Data processing, CSV file I/O (e.g. pd.read_csv)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import subprocess
import re
from nltk.corpus import wordnet
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd /content/drive/MyDrive/Machine learning 2024/Notebooks - Machine Learning/Generic Projects/Fake news classification

### Read WELFake_Dataset.csv from kaggle/input

In [ ]:
# Select the 5000 from 0 and 5000 from 1 rows using the label columns

import pandas as pd
df = pd.read_csv('WELFake_Dataset.csv')
df_0 = df[df['label'] == 0].sample(5000)
df_1 = df[df['label'] == 1].sample(5000)
df = pd.concat([df_0, df_1])


In [ ]:
df.head()

### EDA(Exploratory Data Analysis)

In [ ]:
df.info()

### Missing data analysis

In [ ]:
# Missing data analysis
df.isnull().sum()

In [ ]:
# Drop NA values
df = df.dropna()

# Missing data analysis again
df.isnull().sum()

### Drop unused columns

In [ ]:
df.drop(columns=['Unnamed: 0'],inplace=True)
df.head()

### Class distribution

In [ ]:
# Class distribution
# 0 - Fake, 1 - Real
df['label'].value_counts().plot.pie(autopct='%.2f')

**From the graph we understand that we have balanced data.**

### Define X and y variables

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

In [ ]:
messages = X.copy()

# We have to reset index as we have used dropna() earlier, otherwise it will throw an error
messages.reset_index(inplace=True)

### Lemmatization

In [ ]:
import re
from nltk.corpus import stopwords # “the,” “and,” “is,” “in,” “for,” “where,” “when,...
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer


# Initialize the lemmatizer and stemmer
lm = WordNetLemmatizer()
ps = PorterStemmer()

# Initialize an empty list to store the cleaned messages
corpus = []

# Loop through each message in the 'messages' list
for i in range(len(messages)):
    # Remove non-alphanumeric characters from the message title using regex
    review = re.sub('[^a-zA-Z0-9]', ' ', messages['title'][i])

    # Convert the message to lowercase
    review = review.lower()

    # Split the message into individual words
    review = review.split()

    # Lemmatize, stem each word, and remove stopwords
    review = [lm.lemmatize(x) for x in review if x not in stopwords.words('english')]

    # Join the list of words back into a single string
    review = " ".join(review)

    # Append the cleaned message to the corpus list
    corpus.append(review)

In [ ]:
tf =TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
X=tf.fit_transform(corpus).toarray()


**Explantion**

  Extract the top 5,000 most frequent terms (unigrams, bigrams, and trigrams combined).

  Compute the TF-IDF score for each term across all documents, which represents how important a word is to a document relative to the entire corpus.

  Output a sparse matrix where each row corresponds to a document, and each column corresponds to one of the top 5,000 features (terms), filled with the TF-IDF scores.

  **By combining** these n-grams with the TF-IDF weighting, the vectorizer creates a feature matrix that not only considers the importance of individual words but also the importance of word sequences up to three words long. Limiting to the top 5000 features helps manage the complexity and size of the resulting feature space.

**max_features=5000:** This parameter limits the number of features (or terms) in the TF-IDF matrix to the top 5,000 most frequent terms in the entire corpus. This helps in reducing the dimensionality of the feature space and keeping the most relevant terms.

### Splitting into train and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train,y_train)

### Performance Metrics

In [ ]:
y_pred=rf.predict(X_test)

In [ ]:
# prompt: print accuracy

from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Confusion matrix oluştur
cm = confusion_matrix(y_test, y_pred)

# Matrisi görselleştirme
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, square=True)
plt.xlabel('Predicted Class')
plt.ylabel('Real Class')
plt.show()

In [ ]:
# Apply Naivebayes with all results

import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Naive Bayes modelini oluştur
nb = MultinomialNB()

# Modeli eğit
nb.fit(X_train, y_train)

# Test verileri üzerinde tahmin yap
y_pred_nb = nb.predict(X_test)

# Performans metriklerini hesapla
accuracy = accuracy_score(y_test, y_pred_nb)
precision = precision_score(y_test, y_pred_nb)
recall = recall_score(y_test, y_pred_nb)
f1 = f1_score(y_test, y_pred_nb)

# Sonuçları yazdır
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# Sınıflandırma raporunu yazdır
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))

# Confusion matrix oluştur
cm_nb = confusion_matrix(y_test, y_pred_nb)

# Matrisi görselleştirme
plt.figure(figsize=(8, 6))
sns.heatmap(cm_nb, annot=True, fmt="d", cmap="Blues", cbar=False, square=True)
plt.xlabel('Predicted Class')
plt.ylabel('Real Class')
plt.title('Confusion Matrix for Naive Bayes')
plt.show()


In [ ]:
# Apply all other models in one cell and also ensemble models

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, VotingClassifier
from xgboost import XGBClassifier

# Initialize models
logistic_regression = LogisticRegression()
svm = SVC()
knn = KNeighborsClassifier()
decision_tree = DecisionTreeClassifier()
gradient_boosting = GradientBoostingClassifier()
adaboost = AdaBoostClassifier()
xgboost = XGBClassifier()

# Fit models
logistic_regression.fit(X_train, y_train)
svm.fit(X_train, y_train)
knn.fit(X_train, y_train)
decision_tree.fit(X_train, y_train)
gradient_boosting.fit(X_train, y_train)
adaboost.fit(X_train, y_train)
xgboost.fit(X_train, y_train)

# Predictions
y_pred_lr = logistic_regression.predict(X_test)
y_pred_svm = svm.predict(X_test)
y_pred_knn = knn.predict(X_test)
y_pred_dt = decision_tree.predict(X_test)
y_pred_gb = gradient_boosting.predict(X_test)
y_pred_ada = adaboost.predict(X_test)
y_pred_xgb = xgboost.predict(X_test)

# Evaluate individual models
models = [logistic_regression, svm, knn, decision_tree, gradient_boosting, adaboost, xgboost]
model_names = ["Logistic Regression", "SVM", "KNN", "Decision Tree", "Gradient Boosting", "AdaBoost", "XGBoost"]

for model, name in zip(models, model_names):
    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))
    print("Precision:", precision_score(y_test, model.predict(X_test)))
    print("Recall:", recall_score(y_test, model.predict(X_test)))
    print("F1 Score:", f1_score(y_test, model.predict(X_test)))
    print("\n")

# Ensemble model (Voting Classifier)
ensemble = VotingClassifier(estimators=[
    ('lr', logistic_regression), ('svm', svm), ('knn', knn), ('dt', decision_tree),
    ('gb', gradient_boosting), ('ada', adaboost), ('xgb', xgboost)], voting='hard')
ensemble.fit(X_train, y_train)
y_pred_ensemble = ensemble.predict(X_test)

print("--- Ensemble (Voting Classifier) ---")
print("Accuracy:", accuracy_score(y_test, y_pred_ensemble))
print("Precision:", precision_score(y_test, y_pred_ensemble))
print("Recall:", recall_score(y_test, y_pred_ensemble))
print("F1 Score:", f1_score(y_test, y_pred_ensemble))


--- Logistic Regression ---
Accuracy: 0.8775201612903226
Precision: 0.8723618090452261
Recall: 0.8821138211382114
F1 Score: 0.877210712481051


--- SVM ---
Accuracy: 0.8795362903225806
Precision: 0.8751258811681772
Recall: 0.883130081300813
F1 Score: 0.8791097622660596


--- KNN ---
Accuracy: 0.5534274193548387
Precision: 0.5262593783494105
Recall: 0.9979674796747967
F1 Score: 0.6891228070175438


--- Decision Tree ---
Accuracy: 0.8099798387096774
Precision: 0.8125643666323378
Recall: 0.801829268292683
F1 Score: 0.8071611253196931


--- Gradient Boosting ---
Accuracy: 0.8175403225806451
Precision: 0.768566493955095
Recall: 0.9044715447154471
F1 Score: 0.830999066293184


--- AdaBoost ---
Accuracy: 0.7726814516129032
Precision: 0.7045280122793554
Recall: 0.9329268292682927
F1 Score: 0.8027984258854396


--- XGBoost ---
Accuracy: 0.8482862903225806
Precision: 0.8176744186046512
Recall: 0.8932926829268293
F1 Score: 0.8538125303545411




In [45]:
# Apply stacking model

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB

logistic_regression = LogisticRegression()
knn = KNeighborsClassifier()
decision_tree = DecisionTreeClassifier()
logistic_regression = LogisticRegression()
svm = SVC()
knn = KNeighborsClassifier()
decision_tree = DecisionTreeClassifier()
gradient_boosting = GradientBoostingClassifier()
adaboost = AdaBoostClassifier()
xgboost = XGBClassifier()
nb = MultinomialNB()

# Define base models
base_models = [
    ('lr', logistic_regression),
    ('knn', knn),
    ('dt', decision_tree),
    ('gb', gradient_boosting),
    ('ada', adaboost),
    ('xgb', xgboost),
    ('nb',nb)
]

# Define meta-model
meta_model = LogisticRegression()

# Create stacking classifier
stacking_classifier = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model
)

# Fit the stacking classifier
stacking_classifier.fit(X_train, y_train)

# Make predictions
y_pred_stacking = stacking_classifier.predict(X_test)

In [46]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Evaluate the stacking model
print("--- Stacking Classifier ---")
print("Accuracy:", accuracy_score(y_test, y_pred_stacking))
print("Precision:", precision_score(y_test, y_pred_stacking))
print("Recall:", recall_score(y_test, y_pred_stacking))
print("F1 Score:", f1_score(y_test, y_pred_stacking))

--- Stacking Classifier ---
Accuracy: 0.8795362903225806
Precision: 0.8781725888324873
Recall: 0.8790650406504065
F1 Score: 0.8786185881157947
